# Step 10 — External Validation (GSE25507)

Validate findings on independent dataset GSE25507 (GPL570, 146 blood samples: 82 autism, 64 control).
Two levels: (1) model transfer, (2) independent biomarker validation via 5 XAI methods.

In [ ]:
import sys; sys.path.insert(0, '../src')
import numpy as np, pandas as pd, GEOparse
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import cross_val_score
from imblearn.over_sampling import SMOTE
from asd_pipeline_utils import (
    RANDOM_STATE, SMOTE_PFI_PATH, clean_symbol,
    load_analysis_frame, split_features_and_metadata, get_targets, run_xai_comparison,
)

# ── Load GSE18123 (training) ──
print('Loading GSE18123...')
final_df = load_analysis_frame()
X_train_all, meta_train = split_features_and_metadata(final_df)
y_train_bin, _ = get_targets(meta_train)
print(f'GSE18123: {X_train_all.shape}')

# ── Load & process GSE25507 (validation) ──
print('\nLoading GSE25507...')
gse_val = GEOparse.get_GEO('GSE25507', destdir='./')
expr_val = gse_val.pivot_samples('VALUE').apply(pd.to_numeric, errors='coerce')

# Diagnosis is in 'group' field for this dataset
diag_val = {}
for n, gsm in gse_val.gsms.items():
    for item in gsm.metadata.get('characteristics_ch1', []):
        if ':' in item:
            k, v = item.split(':', 1)
            if k.strip().lower() == 'group':
                diag_val[n] = v.strip()
diag_s = pd.Series(diag_val)
samples = expr_val.columns.intersection(diag_s.index)
print(f'Samples: {len(samples)}, Labels: {diag_s.value_counts().to_dict()}')

# Probe-to-gene mapping (GPL570)
gpl = gse_val.gpls['GPL570'].table
ann = gpl[['ID', 'Gene Symbol']].dropna()
ann['Gene Symbol'] = ann['Gene Symbol'].map(clean_symbol)
ann = ann.dropna()
p2g = dict(zip(ann['ID'].astype(str), ann['Gene Symbol'].astype(str)))

sub = expr_val[samples].copy()
sub.index = sub.index.map(lambda pid: p2g.get(str(pid), np.nan))
sub = sub[sub.index.notna()]
sub = sub.groupby(sub.index).mean()
X_val_all = sub.T  # samples x genes

# Log2 if needed
mx = np.nanmax(X_val_all.to_numpy())
if mx > 50:
    print(f'log2(x+1) applied (max={mx:.0f})')
    X_val_all = np.log2(X_val_all + 1)

y_val_bin = (diag_s.loc[X_val_all.index].str.lower() != 'control').astype(int)
common = X_train_all.columns.intersection(X_val_all.columns)
print(f'\nGSE25507 matrix: {X_val_all.shape}')
print(f'Common genes: {len(common)}')
print(f'PNN present: {"PNN" in common}, PRPF38B: {"PRPF38B" in common}, USP1: {"USP1" in common}')


## Level 1: Model Transfer (train GSE18123 → predict GSE25507)

In [ ]:
pfi_df = pd.read_csv(SMOTE_PFI_PATH)
pfi_genes = pfi_df.head(50)['gene'].tolist()
transfer_genes = [g for g in pfi_genes if g in common]
print(f'PFI genes in GSE25507: {len(transfer_genes)}/{len(pfi_genes)}')
for g in ['PNN', 'PRPF38B', 'USP1']:
    print(f'  {g}: {"YES" if g in transfer_genes else "NO"}')

# Train on GSE18123
imp = SimpleImputer(strategy='median')
sc = StandardScaler()
X_tr = pd.DataFrame(imp.fit_transform(X_train_all[transfer_genes]), index=X_train_all.index, columns=transfer_genes)
X_tr = pd.DataFrame(sc.fit_transform(X_tr), index=X_tr.index, columns=X_tr.columns)
sm = SMOTE(random_state=RANDOM_STATE)
X_tr_r, y_tr_r = sm.fit_resample(X_tr, y_train_bin)
clf = LogisticRegression(solver='saga', max_iter=8000, random_state=RANDOM_STATE)
clf.fit(X_tr_r, y_tr_r)

# Predict on GSE25507
X_te = pd.DataFrame(imp.transform(X_val_all[transfer_genes]), index=X_val_all.index, columns=transfer_genes)
X_te = pd.DataFrame(sc.transform(X_te), index=X_te.index, columns=X_te.columns)
y_pred = clf.predict(X_te)
y_proba = clf.predict_proba(X_te)[:, 1]

print(f'\n=== Model Transfer: GSE18123 -> GSE25507 ===')
print(f'Balanced accuracy: {balanced_accuracy_score(y_val_bin, y_pred):.3f}')
print(f'ROC-AUC: {roc_auc_score(y_val_bin, y_proba):.3f}')
print(f'\n{classification_report(y_val_bin, y_pred, target_names=["Control", "ASD"])}')


## Level 2: Independent Biomarker Validation
Train a NEW model on GSE25507 alone, run 5 XAI methods, check if PNN emerges.

In [ ]:
# Fresh model on GSE25507
sm2 = SMOTE(random_state=RANDOM_STATE)
X_te_r, y_te_r = sm2.fit_resample(X_te, y_val_bin)
clf2 = LogisticRegression(solver='saga', max_iter=8000, random_state=RANDOM_STATE)
clf2.fit(X_te_r, y_te_r)
scores = cross_val_score(clf2, X_te, y_val_bin, cv=5, scoring='balanced_accuracy')
print(f'GSE25507 internal CV: bal_acc = {scores.mean():.3f} +/- {scores.std():.3f}')

# Run 5 XAI methods
print('\nRunning 5 XAI methods on GSE25507...')
rankings_v, top_sets_v, consensus_v = run_xai_comparison(clf2, X_te, y_val_bin, transfer_genes)

top_k = 20
table = pd.DataFrame({
    'Rank': range(1, top_k+1),
    **{n: df.head(top_k)['gene'].values for n, df in rankings_v.items()}
})
print(f'\n=== Top {top_k} by each method ===\n{table.to_string(index=False)}')
print(f'\nConsensus (all 5): {len(consensus_v)} genes -> {sorted(consensus_v)}')


In [ ]:
# ── Does PNN hold up? ──
print('='*60)
print('EXTERNAL VALIDATION SUMMARY')
print('='*60)
for gene in ['PNN', 'PRPF38B', 'USP1']:
    in_methods = sum(1 for s in top_sets_v.values() if gene in s)
    ranks = {}
    for m, df in rankings_v.items():
        r = df[df['gene']==gene]
        ranks[m] = f'#{r.index[0]+1}' if not r.empty else 'N/A'
    status = 'VALIDATED' if in_methods >= 3 else 'PARTIAL' if in_methods >= 1 else 'NOT FOUND'
    print(f'\n{gene}: {status} ({in_methods}/5 methods)')
    print(f'  Ranks: {ranks}')
    print(f'  In consensus: {gene in consensus_v}')
print(f'\nCore genes validated in GSE25507 consensus: {sorted({"PNN","PRPF38B","USP1"} & consensus_v)}')
